# 14 — Temporal alignment & quality

**Research question:** Có thể căn chỉnh dữ liệu không đều theo ngày, giữ provenance và tái tạo snapshot tại một mốc chốt mà không dùng thông tin tương lai không?

Dataset: 193 bản ghi tự tạo, 5 subject giả lập, không có người thật. BodyM không phải dataset lịch sử số đo; không biến photo order thành ngày đo.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
config = json.loads((ROOT / "configs/progress_phase4.json").read_text(encoding="utf-8"))
progress = json.loads((ROOT / "experiments/body_progress_phase4/progress.json").read_text(encoding="utf-8"))


In [2]:
import hashlib
from collections import Counter
from jsonschema import Draft202012Validator
manifest = json.loads((ROOT / "data/manifests/progress_synthetic_v1.json").read_text())
raw_path = ROOT / manifest["input"]
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == manifest["sha256"]
rows = json.loads(raw_path.read_text())
schema = json.loads((ROOT / "schemas/body_progress_observation.schema.json").read_text())
for row in rows:
    Draft202012Validator(schema).validate(row)
print("Dataset:", manifest["dataset_id"], "records:", len(rows), "real subjects:", manifest["real_subjects"])
print("As of:", config["as_of"], "timezone:", config["report_timezone"])
print("States:", dict(Counter(row["analysis_state"] for row in progress["audit_rows"])))

Dataset: body-progress-synthetic-v1 records: 193 real subjects: 0
As of: 2026-10-22T00:00:00+07:00 timezone: Asia/Bangkok
States: {'active': 186, 'duplicate_event': 1, 'superseded': 1, 'duplicate_copy': 1, 'not_yet_recorded': 2, 'quarantined_conflicting_id': 2}


## Baseline / experiment config

Chuẩn hóa kg/lb, cm/in theo registry; giữ giá trị/đơn vị gốc. Timestamp phải có UTC offset; date-only không được gán giờ đo giả, chỉ căn theo lịch khi timezone rõ. Dùng ngày đo cho chuỗi; dùng recorded_at để lọc kiến thức có tại mốc chốt. Không dùng upload time thay capture time. Model mặc định giữ nguyên.

In [3]:
sample = next(row for row in progress["audit_rows"] if row["unit"] == "lb")
display({key: sample[key] for key in ["observation_id", "value", "unit", "canonical_value", "canonical_unit", "day", "provenance"]})
display([{key: row.get(key) for key in ["observation_id", "value", "recorded_at", "analysis_state", "quality_flags"]}
         for row in progress["audit_rows"] if row["analysis_state"] in ["superseded", "not_yet_recorded", "quarantined_conflicting_id"]])

{'observation_id': 'steady-weight-001',
 'value': 176.28162484302808,
 'unit': 'lb',
 'canonical_value': 79.96,
 'canonical_unit': 'kg',
 'day': '2026-08-28',
 'provenance': 'observed'}

[{'observation_id': 'steady-weight-010',
  'value': 900,
  'recorded_at': '2026-09-06T07:05:00+07:00',
  'analysis_state': 'superseded',
  'quality_flags': ['outside_plausibility_bounds']},
 {'observation_id': 'late-revision',
  'value': 10,
  'recorded_at': '2026-10-25T09:00:00+07:00',
  'analysis_state': 'not_yet_recorded',
  'quality_flags': ['not_recorded_as_of']},
 {'observation_id': 'future-reading',
  'value': 20,
  'recorded_at': '2026-10-30T07:05:00+07:00',
  'analysis_state': 'not_yet_recorded',
  'quality_flags': ['future_measurement', 'not_recorded_as_of']},
 {'observation_id': 'conflicting-id',
  'value': 75,
  'recorded_at': '2026-10-16T07:05:00+07:00',
  'analysis_state': 'quarantined_conflicting_id',
  'quality_flags': ['conflicting_observation_id']},
 {'observation_id': 'conflicting-id',
  'value': 85,
  'recorded_at': '2026-10-16T07:05:00+07:00',
  'analysis_state': 'quarantined_conflicting_id',
  'quality_flags': ['conflicting_observation_id']}]

In [4]:
from src.body_progress.engine import build_progress
rebuilt = build_progress(rows, config)
assert rebuilt == progress
assert len(progress["audit_rows"]) == len(rows)
print("Rebuild identical, all raw input records retained in local audit")

Rebuild identical, all raw input records retained in local audit


## Metrics / error analysis / conclusion

ID trùng cùng nội dung chỉ tính một lần; cùng ID khác nội dung bị cách ly. Sửa đổi cần parent cùng sự kiện/nguồn, recorded_at muộn hơn và audit reason/actor; nhánh xung đột cần review, không tự chọn bản mới nhất. Bản ghi đến sau mốc chốt vẫn ở audit nhưng không ảnh hưởng stream, flags hoặc context của snapshot quá khứ.

Unit tests có case bản ghi trùng ID đến muộn, timezone, NaN, missing và inferred cố nhận trạng thái verified. Những kiểm tra này kiểm phần mềm, không xác thực dữ liệu sức khỏe thực tế.